# Application — Mean–variance efficient frontier

Single-period Markowitz on ETF returns. **Not investment advice.**

Run from repo root: `uv run python scripts/run_all.py` first to cache data and figures.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "run_all.py").exists():
    REPO = REPO.parents[1]  # running from python/notebooks/
sys.path.insert(0, str(REPO / "python" / "src"))

from portfolio_linalg.config import load_config
from portfolio_linalg.fetch_data import load_returns
from portfolio_linalg.covariance import build_covariance
from portfolio_linalg.frontier import compute_frontier, min_variance_portfolio
from IPython.display import Image, display, Markdown

In [ ]:
cfg = load_config(REPO / "python" / "config.yaml")
returns = load_returns(cfg)
cov = build_covariance(returns)
frontier = compute_frontier(cov, cfg)
mvp = min_variance_portfolio(cov, cfg)

print(f"Observations: {returns.height}, assets: {len(cov.tickers)}")
print(f"PSD: {cov.is_psd}, min eigenvalue: {cov.min_eigenvalue:.2e}, κ(Σ) ≈ {cov.condition_number:.2e}")
print(f"Min-variance portfolio: μ={mvp['mu']:.4f}, σ={mvp['sigma']:.4f}")

In [ ]:
fig_dir = cfg.figures_dir
for name in ["efficient_frontier.png", "eigenvalues.png", "correlation_heatmap.png", "weights_vs_return.png"]:
    p = fig_dir / name
    if p.exists():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(p)))

## Interpretation (one frontier point)

The **minimum-variance** long-only portfolio balances sample risk \(x^\top \Sigma x\) under \(x \ge 0\), \(\mathbf 1^\top x = 1\). Eigenvalues of \(\Sigma\) describe risk directions; the frontier trades expected return \(\mu^\top x\) for volatility.